In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:13pt;}
div.prompt {min-width:80px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:2px;}
table.dataframe{font-size:2xpt;}
div.text_cell_render h1 {
    color: #001A99 !important;
    font-weight: bold;
}
div.text_cell_render hr {
    border: 0 !important;
    height: 2px !important;
    background-color: #4FC3F7 !important;
}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


# 1. knowledge base 구성을 위한 데이터 생성

In [3]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 12.8 s
Wall time: 37.6 s


180

In [4]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [ ]:
# # 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [6]:
len(embedding.embed_query("샘플"))

3072

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [8]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 15.6 ms
Wall time: 2.01 ms


# 2. 답변 생성을 위한 Retrieval

In [ ]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [ ]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model='gpt-4.1-nano')

In [ ]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [6]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

NameError: name 'retrieved_doc' is not defined

In [7]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

NameError: name 'llm' is not defined

# 4. Langchain

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_upstage import ChatUpstage
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
llm = ChatUpstage(model = 'solar-pro4')
# 2. vector DB load
pc = Pinecone()
index_name = 'tax-index-openai'
vectorStore = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

제공된 문맥은 소득세법 조항만 포함하고 있어 연봉 5천만원에 대한 정확한 소득세 계산 금액(과세표준, 세율, 세액공제 등 반영 결과)을 도출할 수 없으므로 답변드릴 수 없습니다. 실제 소득세는 근로소득공제, 기본공제, 세액공제 등 여러 요소를 적용해야 하며, 정확한 금액은 근로소득 간이세액표 또는 종합소득세 계산을 통해 확정됩니다.
